# Rellenado de Series Temporales (Tarifas Efectivas)
## Fases de Tratado
### Fase 0: Configuración General

El objetivo de este script es procesar la base de datos histórica de Tarifas Efectivas, limpiando valores nulos corruptos (textos "NaN") y construyendo un andamiaje temporal completo (Scaffold). Esto garantiza que cada subpartida tenga un registro continuo mes a mes en la línea de tiempo, rellenando los vacíos (huecos temporales) con nulos reales (`np.nan`) para evitar rupturas en las gráficas o cálculos posteriores.

Dependencias requeridas:
- `pandas (pd):` Manipulación de datos, cruces y manejo de series de tiempo.
- `numpy (np):` Manejo de valores nulos matemáticos.
- `os` / `pathlib (Path):` Manejo seguro de rutas de sistema.

Variables Globales:
- **Rutas:** Directorios de entrada (Raw) y salida (Intermediate), así como los nombres de los archivos correspondientes.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# --- RUTAS DE ARCHIVOS ---
DIR_ENTRADA = "../data/raw"
ARCHIVO_ENTRADA = "tarifa_efectiva_total_hs6_countries.xlsx"
PATH_INPUT_EFECTIVAS = Path(DIR_ENTRADA) / ARCHIVO_ENTRADA

DIR_SALIDA = "../data/intermediate"
ARCHIVO_SALIDA = "Efectivas_HTS_Completo.xlsx"
PATH_OUTPUT_EFECTIVAS = Path(DIR_SALIDA) / ARCHIVO_SALIDA

print("--- CONFIGURACIÓN CARGADA ---")
print(f"Input:  {PATH_INPUT_EFECTIVAS}")
print(f"Output: {PATH_OUTPUT_EFECTIVAS}")

### Fase 0.5: Definición de Funciones

Se encapsula la lógica de transformación de datos para facilitar el mantenimiento:

**1. Funciones Auxiliares (`_nombre`)**
- **`_limpiar_nulos_numericos`**: Itera sobre las columnas numéricas y fuerza su conversión, transformando cadenas de texto basura (como la palabra "NaN") en verdaderos valores nulos (`np.nan`) procesables matemáticamente.
- **`_construir_andamiaje`**: Identifica el rango de fechas global (min/max) y crea un producto cartesiano (MultiIndex) entre todas las subpartidas únicas y todos los meses posibles del rango, generando el "esqueleto" temporal perfecto.

**2. Funciones Principales (`nombre`)**
- **`rellenar_series_temporales`**: Orquestador principal. Carga el archivo desde el disco tolerando formatos (Excel/CSV), ejecuta las limpiezas auxiliares, cruza la base original contra el andamiaje (Left Join) para rellenar los huecos, y ordena el resultado final.

In [2]:
# --- FUNCIONES AUXILIARES ---

def _limpiar_nulos_numericos(df, columnas_excluidas):
    """
    Fuerza la conversión numérica de las columnas de valor, 
    convirtiendo textos como 'NaN' a np.nan.
    """
    cols_a_limpiar = [col for col in df.columns if col not in columnas_excluidas]
    print(f"   Limpiando columnas numéricas: {cols_a_limpiar}")
    
    for col in cols_a_limpiar:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    return df

def _construir_andamiaje(df, col_id, col_fecha):
    """
    Crea un DataFrame base (Scaffold) con la combinación total 
    de todas las subpartidas y todos los meses del rango.
    """
    min_date = df[col_fecha].min()
    max_date = df[col_fecha].max()
    print(f"   Rango temporal detectado: {min_date.date()} a {max_date.date()}")
    
    # Frecuencia 'MS' asegura inicio de mes (Month Start)
    todas_las_fechas = pd.date_range(start=min_date, end=max_date, freq='MS')
    subpartidas_unicas = df[col_id].unique()
    
    # Producto cartesiano
    multi_index = pd.MultiIndex.from_product(
        [subpartidas_unicas, todas_las_fechas], 
        names=[col_id, col_fecha]
    )
    
    return pd.DataFrame(index=multi_index).reset_index()


# --- FUNCIONES PRINCIPALES ---

def rellenar_series_temporales(ruta_input):
    """
    Orquestador: Carga el dataset, limpia NAs, genera el andamiaje,
    hace el cruce (merge) y devuelve el DataFrame consolidado.
    """
    print(f">> Leyendo archivo desde: {ruta_input}")
    try:
        # Intento primario como Excel
        df = pd.read_excel(ruta_input)
    except ValueError:
        # Fallback a CSV si la extensión o formato engaña
        df = pd.read_csv(ruta_input)
    except FileNotFoundError:
        print(f"❌ ERROR: No se encontró el archivo en {ruta_input}")
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ ERROR INESPERADO al leer: {e}")
        return pd.DataFrame()

    # 1. Asegurar formato de fecha
    df['Fecha'] = pd.to_datetime(df['Fecha'])

    # 2. Homogeneización de NAs
    df = _limpiar_nulos_numericos(df, columnas_excluidas=['Fecha', 'Subpartida'])

    # 3. Crear estructura base ininterrumpida
    df_andamiaje = _construir_andamiaje(df, col_id='Subpartida', col_fecha='Fecha')

    # 4. Cruce Left Join para poblar el andamiaje
    # Fechas/Subpartidas sin datos originales quedarán con np.nan perfectos
    print("   Uniendo datos originales con el andamiaje...")
    df_merged = pd.merge(df_andamiaje, df, on=['Subpartida', 'Fecha'], how='left')

    # 5. Ordenamiento final
    df_merged = df_merged.sort_values(by=['Subpartida', 'Fecha'])
    
    return df_merged

### Fase 1: Procesamiento y Rellenado Temporal
Se invoca al orquestador. Se leerá la base de tarifas efectivas cruda, se transformarán los tipos de datos y se expandirá la base para que cada subpartida tenga exactamente la misma cantidad de meses representados.

In [3]:
DF_EFECTIVAS_COMPLETO = rellenar_series_temporales(PATH_INPUT_EFECTIVAS)
print("\n>> ¡Proceso de andamiaje y limpieza terminado exitosamente!")

### Fase 2: Exportación Final
El DataFrame expandido y homogeneizado se guarda en la ruta del directorio intermedio, creando la carpeta si esta no existía previamente.

In [4]:
if not DF_EFECTIVAS_COMPLETO.empty:
    print(f"Generando Excel: {PATH_OUTPUT_EFECTIVAS}...")
    try:
        # Crear carpeta de salida si no existe
        os.makedirs(DIR_SALIDA, exist_ok=True)
        
        # Exportar
        DF_EFECTIVAS_COMPLETO.to_excel(PATH_OUTPUT_EFECTIVAS, index=False)
        
        print("¡ÉXITO! Archivo guardado correctamente. Los NAs ahora son matemáticamente homogéneos.")
        print(f"Total de registros tras el andamiaje: {len(DF_EFECTIVAS_COMPLETO)}")
        print("\nVista previa:")
        print(DF_EFECTIVAS_COMPLETO.head())
        
    except PermissionError:
        print(f"❌ ERROR CRÍTICO: Asegúrate de tener cerrado el archivo '{PATH_OUTPUT_EFECTIVAS}' en Excel.")
    except Exception as e:
        print(f"❌ ERROR AL EXPORTAR: {e}")
else:
    print("⚠️ ADVERTENCIA: El DataFrame resultó vacío, no se generó ningún archivo.")